# Analisis Regresi Logistik Akses Internet IFLS

Notebook ini menganalisis faktor-faktor yang berhubungan dengan `akses_internet` menggunakan regresi logistik biner. Variabel `kelompok_usia` dan `ekonomi_subjektif` tidak digunakan sesuai ketentuan. Variabel identitas `hhid14` dan `pid14` juga tidak dimasukkan sebagai prediktor karena hanya berfungsi sebagai ID responden.

Respon yang dimodelkan adalah:

- `akses_internet = 1`: responden memiliki akses internet
- `akses_internet = 0`: responden tidak memiliki akses internet

Tingkat signifikansi yang digunakan adalah `alpha = 0.05`.


## 0. Persiapan Data

Bagian ini membaca data, memilih variabel analisis, mengatur tipe data, dan memastikan variabel yang dilarang tidak masuk ke model.


In [1]:
options(scipen = 999)
alpha <- 0.05

file_data <- "data_ifls_akses_internet.csv"
data_raw <- read.csv(file_data, stringsAsFactors = FALSE)

response <- "akses_internet"
id_vars <- c("hhid14", "pid14")
excluded_vars <- c("kelompok_usia", "ekonomi_subjektif")

candidate_predictors <- setdiff(names(data_raw), c(id_vars, response, excluded_vars))

cat("Jumlah observasi:", nrow(data_raw), "\n")
cat("Jumlah kolom:", ncol(data_raw), "\n")
cat("Variabel yang tidak digunakan:", paste(excluded_vars, collapse = ", "), "\n")
cat("Prediktor kandidat:", paste(candidate_predictors, collapse = ", "), "\n")

analysis_data <- data_raw[, c(response, candidate_predictors)]
analysis_data[[response]] <- as.integer(analysis_data[[response]])

analysis_data$jenis_kelamin <- factor(
  analysis_data$jenis_kelamin,
  levels = c("Laki-laki", "Perempuan")
)
analysis_data$status_perkawinan <- factor(
  analysis_data$status_perkawinan,
  levels = c("Belum menikah", "Menikah/berpasangan", "Pernah menikah")
)
analysis_data$pendidikan <- factor(
  analysis_data$pendidikan,
  levels = c(
    "Tidak pernah sekolah", "SD/sederajat", "SMP/sederajat",
    "SMA/sederajat", "Perguruan tinggi"
  )
)
analysis_data$mampu_baca_koran <- factor(
  analysis_data$mampu_baca_koran,
  levels = c("Tidak", "Ya")
)
analysis_data$punya_telepon_seluler <- factor(
  analysis_data$punya_telepon_seluler,
  levels = c("Tidak", "Ya")
)
analysis_data$aktivitas_utama <- factor(
  analysis_data$aktivitas_utama,
  levels = c("Bekerja", "Bersekolah", "Mengurus rumah tangga", "Lainnya/tidak bekerja")
)

str(analysis_data)
cat("\nDistribusi akses_internet:\n")
print(table(analysis_data$akses_internet))
cat("\nProporsi akses_internet:\n")
print(round(prop.table(table(analysis_data$akses_internet)), 4))


Jumlah observasi: 31431 


Jumlah kolom: 13 


Variabel yang tidak digunakan: kelompok_usia, ekonomi_subjektif 


Prediktor kandidat: usia, skor_ekonomi_subjektif, jenis_kelamin, status_perkawinan, pendidikan, mampu_baca_koran, punya_telepon_seluler, aktivitas_utama 


'data.frame':	31431 obs. of  9 variables:
 $ akses_internet        : int  0 0 0 1 0 0 0 0 0 0 ...
 $ usia                  : int  59 28 39 16 30 36 26 40 55 54 ...
 $ skor_ekonomi_subjektif: int  3 2 3 3 2 2 2 2 4 3 ...
 $ jenis_kelamin         : Factor w/ 2 levels "Laki-laki","Perempuan": 1 2 2 2 1 1 1 2 1 2 ...
 $ status_perkawinan     : Factor w/ 3 levels "Belum menikah",..: 2 2 2 1 2 2 1 2 2 2 ...
 $ pendidikan            : Factor w/ 5 levels "Tidak pernah sekolah",..: 2 2 2 3 3 2 2 1 2 2 ...
 $ mampu_baca_koran      : Factor w/ 2 levels "Tidak","Ya": 2 2 1 2 2 1 2 1 2 2 ...
 $ punya_telepon_seluler : Factor w/ 2 levels "Tidak","Ya": 1 1 1 2 1 1 2 1 1 1 ...
 $ aktivitas_utama       : Factor w/ 4 levels "Bekerja","Bersekolah",..: 4 3 1 2 1 1 1 3 4 1 ...



Distribusi akses_internet:



    0     1 
20040 11391 



Proporsi akses_internet:



     0      1 
0.6376 0.3624 


<!-- result-narrative -->

### Hasil Persiapan Data untuk Laporan

Dataset yang dianalisis terdiri dari **31.431 observasi** dan **13 kolom**. Variabel respon adalah `akses_internet`, dengan kategori `1` untuk responden yang memiliki akses internet dan `0` untuk responden yang tidak memiliki akses internet. Dari total observasi, sebanyak **20.040 responden (63,76%)** tidak memiliki akses internet dan **11.391 responden (36,24%)** memiliki akses internet.

Sesuai ketentuan analisis, variabel `kelompok_usia` dan `ekonomi_subjektif` tidak digunakan. Variabel `hhid14` dan `pid14` juga tidak dimasukkan ke model karena hanya berperan sebagai identitas. Prediktor yang dianalisis adalah `usia`, `skor_ekonomi_subjektif`, `jenis_kelamin`, `status_perkawinan`, `pendidikan`, `mampu_baca_koran`, `punya_telepon_seluler`, dan `aktivitas_utama`.

Tabel yang perlu ditampilkan pada laporan bagian deskripsi data:

| Komponen | Hasil |
|---|---:|
| Jumlah observasi | 31.431 |
| Tidak memiliki akses internet | 20.040 (63,76%) |
| Memiliki akses internet | 11.391 (36,24%) |
| Variabel respon | `akses_internet` |
| Prediktor yang dikeluarkan | `kelompok_usia`, `ekonomi_subjektif` |


## 1. Uji Independensi pada Faktor-faktor yang Mempengaruhi Y

Untuk prediktor kategorik digunakan uji chi-square independensi antara setiap `X` dan `akses_internet`. Untuk prediktor numerik (`usia` dan `skor_ekonomi_subjektif`) digunakan uji Wilcoxon rank-sum sebagai uji hubungan bivariat terhadap respon biner tanpa mengelompokkan usia.

Hipotesis umum:

- H0: `X` independen terhadap `akses_internet`
- H1: `X` tidak independen terhadap `akses_internet`


In [2]:
fmt_p <- function(p) {
  ifelse(is.na(p), NA, ifelse(p < 0.001, "<0.001", sprintf("%.4f", p)))
}

label_keputusan <- function(p, alpha = 0.05, reject_text = "Tolak H0", keep_text = "Gagal tolak H0") {
  ifelse(is.na(p), NA, ifelse(p < alpha, reject_text, keep_text))
}

uji_independensi <- function(var, data = analysis_data, y = response, alpha = 0.05) {
  x <- data[[var]]
  yy <- data[[y]]

  if (is.numeric(x)) {
    test <- wilcox.test(x ~ yy, exact = FALSE)
    out <- data.frame(
      variabel = var,
      tipe = "Numerik",
      uji = "Wilcoxon rank-sum",
      statistik = unname(test$statistic),
      df = NA_real_,
      min_expected = NA_real_,
      p_value = test$p.value
    )
  } else {
    tab <- table(x, yy)
    chi <- suppressWarnings(chisq.test(tab, correct = FALSE))
    out <- data.frame(
      variabel = var,
      tipe = "Kategorik",
      uji = "Chi-square independensi",
      statistik = unname(chi$statistic),
      df = unname(chi$parameter),
      min_expected = min(chi$expected),
      p_value = chi$p.value
    )
  }

  out$p_value_fmt <- fmt_p(out$p_value)
  out$keputusan <- label_keputusan(
    out$p_value,
    alpha,
    reject_text = "Ada hubungan dengan Y",
    keep_text = "Tidak cukup bukti hubungan"
  )
  out
}

hasil_independensi <- do.call(rbind, lapply(candidate_predictors, uji_independensi))
hasil_independensi[order(hasil_independensi$p_value), ]


,variabel,tipe,uji,statistik,df,min_expected,p_value,p_value_fmt,keputusan
,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
1,usia,Numerik,Wilcoxon rank-sum,188972004.5000,NA,NA,0.000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,<0.001,Ada hubungan dengan Y
4,status_perkawinan,Kategorik,Chi-square independensi,7689.7182,2,863.6300,0.000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,<0.001,Ada hubungan dengan Y
5,pendidikan,Kategorik,Chi-square independensi,9820.5575,4,424.0231,0.000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,<0.001,Ada hubungan dengan Y
7,punya_telepon_seluler,Kategorik,Chi-square independensi,5043.9905,1,2994.2554,0.000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,<0.001,Ada hubungan dengan Y
8,aktivitas_utama,Kategorik,Chi-square independensi,4015.5909,3,681.3363,0.000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,<0.001,Ada hubungan dengan Y
6,mampu_baca_koran,Kategorik,Chi-square independensi,1480.4307,1,899.1464,0.000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000004940656,<0.001,Ada hubungan dengan Y
2,skor_ekonomi_subjektif,Numerik,Wilcoxon rank-sum,90627208.5000,NA,NA,0.000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000002467307845983630759439594881676782733848085626959800720214843750000000000000000000000000000000000000,<0.001,Ada hubungan dengan Y
3,jenis_kelamin,Kategorik,Chi-square independensi,325.1559,1,5336.1676,0.000000000000000000000000000000000000000000000000000000000000000000000001091090109441544066118737199566623985447222366929054260253906250000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,<0.001,Ada hubungan dengan Y


<!-- result-narrative -->

### Hasil Uji Independensi untuk Laporan

Uji independensi menunjukkan bahwa seluruh prediktor memiliki hubungan bivariat yang signifikan dengan `akses_internet` pada taraf signifikansi 5%. Untuk variabel kategorik digunakan uji chi-square, sedangkan untuk variabel numerik digunakan uji Wilcoxon rank-sum karena respon bersifat biner.

Ringkasan hasil uji independensi:

| Variabel | Uji | Statistik | df | p-value | Kesimpulan |
|---|---|---:|---:|---:|---|
| `usia` | Wilcoxon rank-sum | 188.972.004,50 | - | <0,001 | Ada hubungan dengan Y |
| `status_perkawinan` | Chi-square | 7.689,72 | 2 | <0,001 | Ada hubungan dengan Y |
| `pendidikan` | Chi-square | 9.820,56 | 4 | <0,001 | Ada hubungan dengan Y |
| `punya_telepon_seluler` | Chi-square | 5.043,99 | 1 | <0,001 | Ada hubungan dengan Y |
| `aktivitas_utama` | Chi-square | 4.015,59 | 3 | <0,001 | Ada hubungan dengan Y |
| `mampu_baca_koran` | Chi-square | 1.480,43 | 1 | <0,001 | Ada hubungan dengan Y |
| `skor_ekonomi_subjektif` | Wilcoxon rank-sum | 90.627.208,50 | - | <0,001 | Ada hubungan dengan Y |
| `jenis_kelamin` | Chi-square | 325,16 | 1 | <0,001 | Ada hubungan dengan Y |

Dengan demikian, pada analisis awal masing-masing faktor menunjukkan keterkaitan dengan kepemilikan akses internet.


## 2. Uji Signifikansi pada Faktor-faktor yang Mempengaruhi Y (Masing-masing X terhadap Y)

Setiap prediktor diuji dengan regresi logistik bivariat. Nilai p diambil dari likelihood ratio test sehingga prediktor kategorik dengan beberapa level diuji sebagai satu faktor.

Hipotesis:

- H0: prediktor `X` tidak berpengaruh terhadap log odds akses internet
- H1: prediktor `X` berpengaruh terhadap log odds akses internet


In [3]:
uji_logistik_bivariat <- function(var, data = analysis_data, y = response, alpha = 0.05) {
  form <- as.formula(paste(y, "~", var))
  fit <- glm(form, data = data, family = binomial(link = "logit"))
  tab <- drop1(fit, test = "Chisq")

  out <- data.frame(
    variabel = var,
    df = tab[var, "Df"],
    lrt = tab[var, "LRT"],
    p_value = tab[var, "Pr(>Chi)"],
    aic = AIC(fit),
    row.names = NULL
  )
  out$p_value_fmt <- fmt_p(out$p_value)
  out$keputusan <- label_keputusan(
    out$p_value,
    alpha,
    reject_text = "Signifikan",
    keep_text = "Tidak signifikan"
  )
  out
}

hasil_bivariat <- do.call(rbind, lapply(candidate_predictors, uji_logistik_bivariat))
hasil_bivariat[order(hasil_bivariat$p_value), ]


,variabel,df,lrt,p_value,aic,p_value_fmt,keputusan
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
1,usia,1,10660.7726,0.0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,30504.88,<0.001,Signifikan
4,status_perkawinan,2,7753.8732,0.0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,33413.78,<0.001,Signifikan
5,pendidikan,4,11127.0357,0.0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,30044.62,<0.001,Signifikan
6,mampu_baca_koran,1,2182.5480,0.0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,38983.10,<0.001,Signifikan
7,punya_telepon_seluler,1,6317.8706,0.0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,34847.78,<0.001,Signifikan
8,aktivitas_utama,3,4122.4370,0.0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,37047.22,<0.001,Signifikan
2,skor_ekonomi_subjektif,1,949.3528,0.0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000001834106,40216.30,<0.001,Signifikan
3,jenis_kelamin,1,325.1076,0.0000000000000000000000000000000000000000000000000000000000000000000000011178351572921617862422724587467826040665386244654655456542968750000000000000000000000000000000000000000000000000000000000000000000000000000000,40840.54,<0.001,Signifikan


<!-- result-narrative -->

### Hasil Uji Signifikansi Bivariat untuk Laporan

Uji signifikansi masing-masing prediktor terhadap `akses_internet` dilakukan menggunakan regresi logistik bivariat. Hasil likelihood ratio test menunjukkan bahwa seluruh prediktor signifikan secara statistik dengan p-value `<0,001`.

Tabel yang dapat dilaporkan:

| Variabel | df | LRT | p-value | Kesimpulan |
|---|---:|---:|---:|---|
| `usia` | 1 | 10.660,77 | <0,001 | Signifikan |
| `status_perkawinan` | 2 | 7.753,87 | <0,001 | Signifikan |
| `pendidikan` | 4 | 11.127,04 | <0,001 | Signifikan |
| `mampu_baca_koran` | 1 | 2.182,55 | <0,001 | Signifikan |
| `punya_telepon_seluler` | 1 | 6.317,87 | <0,001 | Signifikan |
| `aktivitas_utama` | 3 | 4.122,44 | <0,001 | Signifikan |
| `skor_ekonomi_subjektif` | 1 | 949,35 | <0,001 | Signifikan |
| `jenis_kelamin` | 1 | 325,11 | <0,001 | Signifikan |

Hasil ini menunjukkan bahwa secara terpisah, setiap faktor memiliki kontribusi yang bermakna dalam menjelaskan peluang responden memiliki akses internet.


## 3. Uji Korelasi pada Faktor yang Mempengaruhi Y

Bagian ini memeriksa asosiasi antar-prediktor sebelum model multivariat. Karena prediktor terdiri dari campuran numerik dan kategorik, ukuran asosiasi disesuaikan dengan tipe pasangan variabel:

- Numerik vs numerik: korelasi Spearman absolut
- Kategorik vs kategorik: Cramer's V
- Numerik vs kategorik: eta/correlation ratio

Nilai mendekati 0 menunjukkan asosiasi lemah, sedangkan nilai mendekati 1 menunjukkan asosiasi kuat. Pasangan dengan nilai asosiasi tinggi perlu diperhatikan karena dapat mengindikasikan redundansi informasi antar-prediktor.


In [4]:
cramers_v <- function(x, y) {
  tab <- table(x, y)
  if (min(dim(tab)) < 2) return(NA_real_)
  chi <- suppressWarnings(chisq.test(tab, correct = FALSE))
  n <- sum(tab)
  sqrt(unname(chi$statistic) / (n * (min(dim(tab)) - 1)))
}

correlation_ratio <- function(categories, values) {
  categories <- as.factor(categories)
  values <- as.numeric(values)
  grand_mean <- mean(values, na.rm = TRUE)
  ss_between <- sum(tapply(values, categories, function(v) length(v) * (mean(v, na.rm = TRUE) - grand_mean)^2))
  ss_total <- sum((values - grand_mean)^2, na.rm = TRUE)
  if (ss_total == 0) return(NA_real_)
  sqrt(ss_between / ss_total)
}

asosiasi_pair <- function(var1, var2, data = analysis_data) {
  x <- data[[var1]]
  y <- data[[var2]]

  if (is.numeric(x) && is.numeric(y)) {
    return(abs(cor(x, y, method = "spearman", use = "complete.obs")))
  }
  if (!is.numeric(x) && !is.numeric(y)) {
    return(cramers_v(x, y))
  }
  if (is.numeric(x) && !is.numeric(y)) {
    return(correlation_ratio(y, x))
  }
  correlation_ratio(x, y)
}

assoc_matrix <- matrix(
  NA_real_,
  nrow = length(candidate_predictors),
  ncol = length(candidate_predictors),
  dimnames = list(candidate_predictors, candidate_predictors)
)

for (i in seq_along(candidate_predictors)) {
  for (j in seq_along(candidate_predictors)) {
    assoc_matrix[i, j] <- if (i == j) 1 else asosiasi_pair(candidate_predictors[i], candidate_predictors[j])
  }
}

round(assoc_matrix, 3)

assoc_pairs <- data.frame()
for (i in seq_along(candidate_predictors)) {
  for (j in seq_along(candidate_predictors)) {
    if (j > i) {
      assoc_pairs <- rbind(
        assoc_pairs,
        data.frame(
          variabel_1 = candidate_predictors[i],
          variabel_2 = candidate_predictors[j],
          asosiasi = assoc_matrix[i, j]
        )
      )
    }
  }
}

cat("\nPasangan prediktor dengan asosiasi >= 0.70:\n")
strong_pairs <- subset(assoc_pairs, asosiasi >= 0.70)
if (nrow(strong_pairs) == 0) {
  cat("Tidak ada pasangan prediktor dengan asosiasi >= 0.70.\n")
} else {
  print(strong_pairs[order(-strong_pairs$asosiasi), ])
}


,usia,skor_ekonomi_subjektif,jenis_kelamin,status_perkawinan,pendidikan,mampu_baca_koran,punya_telepon_seluler,aktivitas_utama
usia,1.000,0.091,0.019,0.595,0.477,0.342,0.418,0.421
skor_ekonomi_subjektif,0.091,1.000,0.078,0.099,0.234,0.110,0.141,0.101
jenis_kelamin,0.019,0.078,1.000,0.171,0.095,0.078,0.147,0.507
status_perkawinan,0.595,0.099,0.171,1.000,0.250,0.213,0.257,0.446
pendidikan,0.477,0.234,0.095,0.250,1.000,0.642,0.501,0.149
mampu_baca_koran,0.342,0.110,0.078,0.213,0.642,1.000,0.374,0.097
punya_telepon_seluler,0.418,0.141,0.147,0.257,0.501,0.374,1.000,0.170
aktivitas_utama,0.421,0.101,0.507,0.446,0.149,0.097,0.170,1.000



Pasangan prediktor dengan asosiasi >= 0.70:


Tidak ada pasangan prediktor dengan asosiasi >= 0.70.


<!-- result-narrative -->

### Hasil Uji Korelasi/Asosiasi antar Faktor untuk Laporan

Pemeriksaan asosiasi antar-prediktor dilakukan untuk melihat apakah terdapat hubungan yang terlalu kuat antar faktor penjelas. Ukuran asosiasi yang digunakan disesuaikan dengan tipe variabel: Spearman untuk numerik-numerik, Cramer's V untuk kategorik-kategorik, dan correlation ratio untuk numerik-kategorik.

Tidak ditemukan pasangan prediktor dengan nilai asosiasi `>= 0,70`. Artinya, berdasarkan batas praktis tersebut, tidak ada indikasi asosiasi yang terlalu tinggi antar-prediktor. Beberapa asosiasi terbesar yang perlu dicatat adalah:

| Pasangan Variabel | Nilai Asosiasi | Catatan |
|---|---:|---|
| `pendidikan` dan `mampu_baca_koran` | 0,642 | Cukup kuat, tetapi masih di bawah 0,70 |
| `usia` dan `status_perkawinan` | 0,595 | Cukup kuat, wajar secara substantif |
| `jenis_kelamin` dan `aktivitas_utama` | 0,507 | Sedang |
| `pendidikan` dan `punya_telepon_seluler` | 0,501 | Sedang |
| `usia` dan `pendidikan` | 0,477 | Sedang |

Kesimpulannya, seluruh prediktor masih dapat dipertimbangkan dalam model regresi logistik multivariat karena tidak ada pasangan faktor dengan asosiasi sangat tinggi.


<!-- spearman-x-y -->

### 3b. Uji Korelasi Spearman Masing-masing X terhadap Y

Selain memeriksa hubungan antar-prediktor, bagian ini menghitung korelasi Spearman antara setiap prediktor `X` dan `akses_internet`. Karena `akses_internet` berbentuk biner (`0` dan `1`), korelasi Spearman di sini digunakan sebagai ukuran hubungan monoton antara setiap faktor dengan peluang memiliki akses internet.

Untuk prediktor numerik, nilai asli digunakan. Untuk prediktor kategorik, kategori dikodekan menjadi angka sesuai urutan level faktor yang telah ditetapkan pada bagian persiapan data. Karena itu, arah korelasi pada variabel kategorik nominal perlu dibaca hati-hati: tanda positif atau negatif mengikuti urutan kode kategori, bukan selalu makna substantif alami.


In [5]:
spearman_xy <- function(var, data = analysis_data, y = response, alpha = 0.05) {
  x <- data[[var]]
  yy <- as.numeric(data[[y]])

  if (is.numeric(x)) {
    x_num <- as.numeric(x)
    coding <- "nilai asli"
  } else {
    x_num <- as.numeric(x)
    coding <- paste(paste(levels(x), seq_along(levels(x)), sep = "="), collapse = "; ")
  }

  test <- suppressWarnings(cor.test(x_num, yy, method = "spearman", exact = FALSE))

  out <- data.frame(
    variabel = var,
    tipe = ifelse(is.numeric(x), "Numerik", "Kategorik dikodekan"),
    rho_spearman = unname(test$estimate),
    p_value = test$p.value,
    p_value_fmt = fmt_p(test$p.value),
    keputusan = label_keputusan(
      test$p.value,
      alpha,
      reject_text = "Berkorelasi dengan Y",
      keep_text = "Tidak cukup bukti korelasi"
    ),
    pengkodean = coding,
    row.names = NULL
  )
  out
}

hasil_spearman_xy <- do.call(rbind, lapply(candidate_predictors, spearman_xy))
hasil_spearman_xy <- hasil_spearman_xy[order(-abs(hasil_spearman_xy$rho_spearman)), ]
hasil_spearman_xy


,variabel,tipe,rho_spearman,p_value,p_value_fmt,keputusan,pengkodean
,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>
5,pendidikan,Kategorik dikodekan,0.5563145,0.0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,<0.001,Berkorelasi dengan Y,Tidak pernah sekolah=1; SD/sederajat=2; SMP/sederajat=3; SMA/sederajat=4; Perguruan tinggi=5
1,usia,Numerik,-0.5460132,0.0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,<0.001,Berkorelasi dengan Y,nilai asli
4,status_perkawinan,Kategorik dikodekan,-0.4690848,0.0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,<0.001,Berkorelasi dengan Y,Belum menikah=1; Menikah/berpasangan=2; Pernah menikah=3
7,punya_telepon_seluler,Kategorik dikodekan,0.4005973,0.0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,<0.001,Berkorelasi dengan Y,Tidak=1; Ya=2
6,mampu_baca_koran,Kategorik dikodekan,0.2170276,0.0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,<0.001,Berkorelasi dengan Y,Tidak=1; Ya=2
2,skor_ekonomi_subjektif,Numerik,0.1831750,0.0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000002949114,<0.001,Berkorelasi dengan Y,nilai asli
3,jenis_kelamin,Kategorik dikodekan,-0.1017107,0.0000000000000000000000000000000000000000000000000000000000000000000000004727664225354015610242736089574577817984391003847122192382812500000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,<0.001,Berkorelasi dengan Y,Laki-laki=1; Perempuan=2
8,aktivitas_utama,Kategorik dikodekan,-0.0353492,0.0000000003640855386211461837365696569968065432476578280329704284667968750000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,<0.001,Berkorelasi dengan Y,Bekerja=1; Bersekolah=2; Mengurus rumah tangga=3; Lainnya/tidak bekerja=4


<!-- spearman-x-y -->

### Hasil Korelasi Spearman X terhadap Y untuk Laporan

Tabel Spearman X terhadap Y digunakan sebagai pelengkap tahap korelasi. Hasil ini menunjukkan kekuatan dan arah hubungan monoton sederhana antara masing-masing prediktor dan `akses_internet` sebelum masuk ke model regresi logistik multivariat.

Interpretasi umum:

- `rho_spearman > 0`: nilai/kode X yang lebih tinggi cenderung berhubungan dengan `akses_internet = 1`.
- `rho_spearman < 0`: nilai/kode X yang lebih tinggi cenderung berhubungan dengan `akses_internet = 0`.
- Nilai absolut rho yang makin besar menunjukkan hubungan monoton yang makin kuat.

Catatan untuk laporan: untuk variabel kategorik nominal seperti `jenis_kelamin` dan `aktivitas_utama`, tanda korelasi mengikuti urutan pengkodean kategori. Karena itu, hasil Spearman pada variabel nominal sebaiknya dipakai sebagai ukuran asosiasi tambahan, sedangkan kesimpulan utama tetap mengacu pada uji chi-square dan regresi logistik.


## 4. Uji Serentak pada Faktor yang Mempengaruhi Y

Model penuh memasukkan seluruh prediktor kandidat kecuali `kelompok_usia` dan `ekonomi_subjektif`. Uji serentak dilakukan dengan likelihood ratio test antara model kosong dan model penuh.

Hipotesis:

- H0: semua koefisien prediktor sama dengan 0
- H1: minimal ada satu koefisien prediktor yang tidak sama dengan 0


In [6]:
full_formula <- as.formula(paste(response, "~", paste(candidate_predictors, collapse = " + ")))
model_null <- glm(as.formula(paste(response, "~ 1")), data = analysis_data, family = binomial(link = "logit"))
model_full <- glm(full_formula, data = analysis_data, family = binomial(link = "logit"))

uji_serentak <- anova(model_null, model_full, test = "Chisq")
print(uji_serentak)

p_serentak <- uji_serentak$`Pr(>Chi)`[2]
cat("\nP-value uji serentak:", fmt_p(p_serentak), "\n")
cat("Keputusan:", label_keputusan(p_serentak, alpha, "Model penuh signifikan", "Model penuh tidak signifikan"), "\n")


Analysis of Deviance Table

Model 1: akses_internet ~ 1
Model 2: akses_internet ~ usia + skor_ekonomi_subjektif + jenis_kelamin + 
    status_perkawinan + pendidikan + mampu_baca_koran + punya_telepon_seluler + 
    aktivitas_utama
  Resid. Df Resid. Dev Df Deviance              Pr(>Chi)    
1     31430      41162                                      
2     31416      20350 14    20812 < 0.00000000000000022 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1



P-value uji serentak: <0.001 


Keputusan: Model penuh signifikan 


<!-- result-narrative -->

### Hasil Uji Serentak untuk Laporan

Uji serentak membandingkan model kosong dengan model penuh yang memuat seluruh prediktor kandidat. Hasil analisis deviance menunjukkan nilai deviance sebesar **20.812** dengan df sebesar **14** dan p-value `<0,001`.

| Model | Residual df | Residual deviance | df | Deviance | p-value |
|---|---:|---:|---:|---:|---:|
| Model kosong | 31.430 | 41.162 | - | - | - |
| Model penuh | 31.416 | 20.350 | 14 | 20.812 | <0,001 |

Karena p-value `<0,05`, H0 ditolak. Artinya, secara serentak minimal terdapat satu prediktor yang berpengaruh signifikan terhadap peluang responden memiliki akses internet. Model penuh lebih baik dibandingkan model tanpa prediktor.


## 5. Uji Individu pada Faktor-faktor yang Mempengaruhi Y

Uji individu dilakukan pada model penuh menggunakan joint Wald test per prediktor. Untuk prediktor kategorik, pengujian dilakukan terhadap faktor secara keseluruhan, bukan hanya salah satu levelnya.


In [7]:
uji_wald_term <- function(model, term) {
  X <- model.matrix(model)
  assign_id <- attr(X, "assign")
  term_labels <- attr(terms(model), "term.labels")
  term_id <- match(term, term_labels)
  coef_names <- colnames(X)[assign_id == term_id]

  beta <- coef(model)[coef_names]
  beta <- beta[!is.na(beta)]
  V <- vcov(model)[names(beta), names(beta), drop = FALSE]

  V_inv_beta <- tryCatch(
    solve(V, beta),
    error = function(e) qr.solve(V, beta)
  )
  wald <- as.numeric(t(beta) %*% V_inv_beta)
  df <- length(beta)
  p_value <- pchisq(wald, df = df, lower.tail = FALSE)

  data.frame(
    variabel = term,
    df = df,
    wald = wald,
    p_value = p_value,
    row.names = NULL
  )
}

hasil_individu <- do.call(rbind, lapply(candidate_predictors, function(x) {
  uji_wald_term(model_full, x)
}))
hasil_individu$p_value_fmt <- fmt_p(hasil_individu$p_value)
hasil_individu$keputusan <- label_keputusan(
  hasil_individu$p_value,
  alpha,
  reject_text = "Signifikan dalam model penuh",
  keep_text = "Tidak signifikan dalam model penuh"
)

hasil_individu[order(hasil_individu$p_value), ]

cat("\nRingkasan koefisien model penuh:\n")
print(coef(summary(model_full)))


,variabel,df,wald,p_value,p_value_fmt,keputusan
,<chr>,<int>,<dbl>,<dbl>,<chr>,<chr>
1,usia,1,2250.03027,0.0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,<0.001,Signifikan dalam model penuh
5,pendidikan,4,3509.41230,0.0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,<0.001,Signifikan dalam model penuh
7,punya_telepon_seluler,1,753.14291,0.0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000008318067,<0.001,Signifikan dalam model penuh
4,status_perkawinan,2,388.57549,0.0000000000000000000000000000000000000000000000000000000000000000000000000000000000004187003905566761176402362787385413867013994604349136352539062500000000000000000000000000,<0.001,Signifikan dalam model penuh
3,jenis_kelamin,1,132.15668,0.0000000000000000000000000000013826448764489504516736473327398471155902370810508728027343750000000000000000000000000000000000000000000000000000000000000000000000000000000000,<0.001,Signifikan dalam model penuh
2,skor_ekonomi_subjektif,1,110.92039,0.0000000000000000000000000615950869267902984417056844179683139373082667589187622070312500000000000000000000000000000000000000000000000000000000000000000000000000000000000000,<0.001,Signifikan dalam model penuh
8,aktivitas_utama,3,42.10642,0.0000000038088201680575431598389790632808171721990220248699188232421875000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,<0.001,Signifikan dalam model penuh
6,mampu_baca_koran,1,31.63957,0.0000000185608624241059380058013750236511896218871697783470153808593750000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,<0.001,Signifikan dalam model penuh



Ringkasan koefisien model penuh:


                                       Estimate Std. Error     z value
(Intercept)                          -1.9221138 0.50374998  -3.8156106
usia                                 -0.1076753 0.00226998 -47.4344839
skor_ekonomi_subjektif                0.2213285 0.02101511  10.5318751
jenis_kelaminPerempuan               -0.5000849 0.04350100 -11.4959419
status_perkawinanMenikah/berpasangan -1.1737573 0.06116231 -19.1908616
status_perkawinanPernah menikah      -0.6823509 0.10984190  -6.2121179
pendidikanSD/sederajat                0.3566983 0.42832332   0.8327781
pendidikanSMP/sederajat               1.0987084 0.42744797   2.5703910
pendidikanSMA/sederajat               2.3813310 0.42681379   5.5793207
pendidikanPerguruan tinggi            4.2310022 0.42914899   9.8590520
mampu_baca_koranYa                    1.7644985 0.31369384   5.6249064
punya_telepon_selulerYa               2.0658641 0.07527713  27.4434492
aktivitas_utamaBersekolah            -0.1814709 0.09415547  -1.9273540
aktivi

<!-- result-narrative -->

### Hasil Uji Individu untuk Laporan

Uji individu pada model penuh dilakukan dengan joint Wald test untuk setiap prediktor. Hasilnya menunjukkan bahwa semua prediktor signifikan dalam model penuh pada taraf signifikansi 5%.

| Variabel | df | Wald | p-value | Kesimpulan |
|---|---:|---:|---:|---|
| `usia` | 1 | 2.250,03 | <0,001 | Signifikan |
| `pendidikan` | 4 | 3.509,41 | <0,001 | Signifikan |
| `punya_telepon_seluler` | 1 | 753,14 | <0,001 | Signifikan |
| `status_perkawinan` | 2 | 388,58 | <0,001 | Signifikan |
| `jenis_kelamin` | 1 | 132,16 | <0,001 | Signifikan |
| `skor_ekonomi_subjektif` | 1 | 110,92 | <0,001 | Signifikan |
| `aktivitas_utama` | 3 | 42,11 | <0,001 | Signifikan |
| `mampu_baca_koran` | 1 | 31,64 | <0,001 | Signifikan |

Catatan penting untuk laporan: uji ini menilai signifikansi faktor secara keseluruhan. Untuk variabel kategorik seperti `pendidikan` dan `aktivitas_utama`, suatu faktor dapat signifikan secara keseluruhan meskipun ada level tertentu yang koefisiennya tidak signifikan dibanding kategori referensi.


## 6. Hasil Uji Individu: X yang Signifikan Dimodelkan Ulang

Prediktor dengan nilai p `< alpha` pada uji individu model penuh dipilih untuk membentuk model akhir. Jika tidak ada prediktor signifikan, model akhir akan kembali ke model kosong.


In [8]:
significant_predictors <- hasil_individu$variabel[hasil_individu$p_value < alpha]
cat("Prediktor signifikan:", paste(significant_predictors, collapse = ", "), "\n")

if (length(significant_predictors) == 0) {
  final_formula <- as.formula(paste(response, "~ 1"))
} else {
  final_formula <- as.formula(paste(response, "~", paste(significant_predictors, collapse = " + ")))
}

model_final <- glm(final_formula, data = analysis_data, family = binomial(link = "logit"))
cat("Formula model akhir:\n")
print(final_formula)
cat("\nRingkasan model akhir:\n")
print(summary(model_final))

cat("\nPerbandingan AIC:\n")
print(data.frame(
  model = c("Null", "Penuh", "Akhir signifikan"),
  AIC = c(AIC(model_null), AIC(model_full), AIC(model_final))
))


Prediktor signifikan: usia, skor_ekonomi_subjektif, jenis_kelamin, status_perkawinan, pendidikan, mampu_baca_koran, punya_telepon_seluler, aktivitas_utama 


Formula model akhir:


akses_internet ~ usia + skor_ekonomi_subjektif + jenis_kelamin + 
    status_perkawinan + pendidikan + mampu_baca_koran + punya_telepon_seluler + 
    aktivitas_utama



Ringkasan model akhir:



Call:
glm(formula = final_formula, family = binomial(link = "logit"), 
    data = analysis_data)

Coefficients:
                                     Estimate Std. Error z value
(Intercept)                          -1.92211    0.50375  -3.816
usia                                 -0.10768    0.00227 -47.434
skor_ekonomi_subjektif                0.22133    0.02101  10.532
jenis_kelaminPerempuan               -0.50009    0.04350 -11.496
status_perkawinanMenikah/berpasangan -1.17376    0.06116 -19.191
status_perkawinanPernah menikah      -0.68235    0.10984  -6.212
pendidikanSD/sederajat                0.35670    0.42832   0.833
pendidikanSMP/sederajat               1.09871    0.42745   2.570
pendidikanSMA/sederajat               2.38133    0.42681   5.579
pendidikanPerguruan tinggi            4.23100    0.42915   9.859
mampu_baca_koranYa                    1.76450    0.31369   5.625
punya_telepon_selulerYa               2.06586    0.07528  27.443
aktivitas_utamaBersekolah            -0.18


Perbandingan AIC:


             model      AIC
1             Null 41163.65
2            Penuh 20379.61
3 Akhir signifikan 20379.61


<!-- result-narrative -->

### Hasil Pemodelan Ulang untuk Laporan

Karena seluruh prediktor signifikan pada uji individu berbasis joint Wald test, model akhir memuat semua prediktor kandidat, yaitu `usia`, `skor_ekonomi_subjektif`, `jenis_kelamin`, `status_perkawinan`, `pendidikan`, `mampu_baca_koran`, `punya_telepon_seluler`, dan `aktivitas_utama`.

Model akhir dapat ditulis sebagai:

```text
logit(P(akses_internet = 1)) = beta0 + beta1 usia + beta2 skor_ekonomi_subjektif
                              + beta3 jenis_kelamin + beta4 status_perkawinan
                              + beta5 pendidikan + beta6 mampu_baca_koran
                              + beta7 punya_telepon_seluler + beta8 aktivitas_utama
```

Nilai AIC model kosong adalah **41.163,65**, sedangkan AIC model akhir adalah **20.379,61**. Penurunan AIC yang besar menunjukkan bahwa model dengan prediktor memiliki kualitas penjelasan yang jauh lebih baik dibanding model kosong.


## 7. Peluang

Peluang yang dihitung adalah peluang responden memiliki akses internet, yaitu `P(akses_internet = 1 | X)`, berdasarkan model akhir.


In [ ]:
analysis_data$peluang_akses_internet <- predict(model_final, type = "response")

cat("Ringkasan peluang prediksi model akhir:\n")
print(summary(analysis_data$peluang_akses_internet))

cat("\nRata-rata peluang prediksi menurut status aktual Y:\n")
print(aggregate(peluang_akses_internet ~ akses_internet, data = analysis_data, mean))

tabel_peluang_laporan <- head(
  analysis_data[, c(response, significant_predictors, "peluang_akses_internet")],
  10
)
tabel_peluang_laporan$No <- seq_len(nrow(tabel_peluang_laporan))
tabel_peluang_laporan <- tabel_peluang_laporan[, c("No", response, significant_predictors, "peluang_akses_internet")]

names(tabel_peluang_laporan) <- c(
  "No",
  "Akses Internet Aktual",
  "Usia",
  "Skor Ekonomi Subjektif",
  "Jenis Kelamin",
  "Status Perkawinan",
  "Pendidikan",
  "Mampu Baca Koran",
  "Punya Telepon Seluler",
  "Aktivitas Utama",
  "Peluang Akses Internet"
)

tabel_peluang_laporan[["Peluang Akses Internet"]] <- round(
  tabel_peluang_laporan[["Peluang Akses Internet"]],
  10
)

write.csv2(tabel_peluang_laporan, "tabel_peluang_10_observasi.csv", row.names = FALSE)

cat("\nTabel peluang prediksi untuk laporan (10 observasi pertama):\n")
tabel_peluang_laporan


<!-- result-narrative -->

### Hasil Peluang Prediksi untuk Laporan

Peluang prediksi dari model akhir menggambarkan estimasi probabilitas responden memiliki akses internet berdasarkan karakteristiknya. Rata-rata peluang prediksi adalah **0,3624**, konsisten dengan proporsi aktual responden yang memiliki akses internet.

Ringkasan peluang prediksi:

| Statistik | Nilai |
|---|---:|
| Minimum | 0,000003 |
| Kuartil 1 | 0,0186 |
| Median | 0,2291 |
| Mean | 0,3624 |
| Kuartil 3 | 0,7053 |
| Maksimum | 0,9947 |

Rata-rata peluang prediksi untuk responden yang secara aktual tidak memiliki akses internet adalah **0,1621**, sedangkan untuk responden yang secara aktual memiliki akses internet adalah **0,7148**. Perbedaan ini menunjukkan bahwa model cukup mampu membedakan kelompok dengan dan tanpa akses internet.


## 8. Odds Ratio

Odds ratio diperoleh dari `exp(koefisien)`. Nilai OR lebih dari 1 berarti kategori/nilai prediktor tersebut meningkatkan odds akses internet dibanding referensi, sedangkan OR kurang dari 1 berarti menurunkan odds akses internet dibanding referensi, dengan asumsi prediktor lain konstan.


In [10]:
buat_or_table <- function(model) {
  koef <- coef(summary(model))
  ci <- confint.default(model)
  out <- data.frame(
    term = rownames(koef),
    estimate = koef[, "Estimate"],
    std_error = koef[, "Std. Error"],
    z_value = koef[, "z value"],
    p_value = koef[, "Pr(>|z|)"],
    OR = exp(koef[, "Estimate"]),
    CI_95_low = exp(ci[, 1]),
    CI_95_high = exp(ci[, 2]),
    row.names = NULL
  )
  out$p_value_fmt <- fmt_p(out$p_value)
  out
}

or_model_akhir <- buat_or_table(model_final)
or_model_akhir <- subset(or_model_akhir, term != "(Intercept)")
or_model_akhir[order(or_model_akhir$p_value), ]


,term,estimate,std_error,z_value,p_value,OR,CI_95_low,CI_95_high,p_value_fmt
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
2,usia,-0.1076753,0.00226998,-47.4344839,0.0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0.8979191,0.8939331,0.9019229,<0.001
12,punya_telepon_selulerYa,2.0658641,0.07527713,27.4434492,0.0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000008318067,7.8921144,6.8095336,9.1468042,<0.001
5,status_perkawinanMenikah/berpasangan,-1.1737573,0.06116231,-19.1908616,0.0000000000000000000000000000000000000000000000000000000000000000000000000000000004412628068846246565202356970658570389787200838327407836914062500000000000000000000000000000,0.3092030,0.2742725,0.3485821,<0.001
4,jenis_kelaminPerempuan,-0.5000849,0.04350100,-11.4959419,0.0000000000000000000000000000013826448764489308330440098249169977862038649618625640869140625000000000000000000000000000000000000000000000000000000000000000000000000000000000,0.6064792,0.5569135,0.6604562,<0.001
3,skor_ekonomi_subjektif,0.2213285,0.02101511,10.5318751,0.0000000000000000000000000615950869267907920485971562385429933783598244190216064453125000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,1.2477332,1.1973845,1.3001990,<0.001
10,pendidikanPerguruan tinggi,4.2310022,0.42914899,9.8590520,0.0000000000000000000000626379943535317540524026691528547416965011507272720336914062500000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,68.7861354,29.6625598,159.5119385,<0.001
6,status_perkawinanPernah menikah,-0.6823509,0.10984190,-6.2121179,0.0000000005227516448661583162854649131645601300988346338272094726562500000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0.5054274,0.4075315,0.6268395,<0.001
14,aktivitas_utamaMengurus rumah tangga,-0.3030868,0.04954953,-6.1168452,0.0000000009544599614469455397397501528189422970172017812728881835937500000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0.7385350,0.6701846,0.8138563,<0.001
11,mampu_baca_koranYa,1.7644985,0.31369384,5.6249064,0.0000000185608624241060041811651748488998237007763236761093139648437500000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,5.8386433,3.1571504,10.7976345,<0.001


<!-- result-narrative -->

### Hasil Odds Ratio untuk Laporan

Odds ratio menunjukkan perubahan odds memiliki akses internet ketika prediktor berubah satu unit atau berpindah kategori, dengan prediktor lain dianggap konstan. Beberapa hasil utama adalah:

| Variabel/Level | OR | 95% CI | p-value | Interpretasi Singkat |
|---|---:|---:|---:|---|
| `usia` | 0,898 | 0,894 - 0,902 | <0,001 | Setiap kenaikan 1 tahun usia menurunkan odds akses internet sekitar 10,2% |
| `skor_ekonomi_subjektif` | 1,248 | 1,197 - 1,300 | <0,001 | Setiap kenaikan 1 skor ekonomi subjektif meningkatkan odds sekitar 24,8% |
| `jenis_kelamin`: Perempuan | 0,606 | 0,557 - 0,660 | <0,001 | Perempuan memiliki odds lebih rendah dibanding laki-laki |
| `status_perkawinan`: Menikah/berpasangan | 0,309 | 0,274 - 0,349 | <0,001 | Odds lebih rendah dibanding belum menikah |
| `status_perkawinan`: Pernah menikah | 0,505 | 0,408 - 0,627 | <0,001 | Odds lebih rendah dibanding belum menikah |
| `pendidikan`: SMP/sederajat | 3,000 | 1,298 - 6,934 | 0,0102 | Odds lebih tinggi dibanding tidak pernah sekolah |
| `pendidikan`: SMA/sederajat | 10,819 | 4,687 - 24,975 | <0,001 | Odds jauh lebih tinggi dibanding tidak pernah sekolah |
| `pendidikan`: Perguruan tinggi | 68,786 | 29,663 - 159,512 | <0,001 | Odds sangat lebih tinggi dibanding tidak pernah sekolah |
| `mampu_baca_koran`: Ya | 5,839 | 3,157 - 10,798 | <0,001 | Odds lebih tinggi dibanding tidak mampu membaca koran |
| `punya_telepon_seluler`: Ya | 7,892 | 6,810 - 9,147 | <0,001 | Odds lebih tinggi dibanding tidak punya telepon seluler |
| `aktivitas_utama`: Mengurus rumah tangga | 0,739 | 0,670 - 0,814 | <0,001 | Odds lebih rendah dibanding bekerja |

Level yang tidak signifikan pada taraf 5% dalam tabel koefisien adalah `pendidikan: SD/sederajat` (p = 0,4050), `aktivitas_utama: Bersekolah` (p = 0,0539), dan `aktivitas_utama: Lainnya/tidak bekerja` (p = 0,1432). Namun, faktor `pendidikan` dan `aktivitas_utama` tetap signifikan secara keseluruhan pada uji individu model penuh.


## 9. Uji Kesesuaian Model

Kesesuaian model dinilai menggunakan uji Hosmer-Lemeshow berbasis 10 grup peluang prediksi. Selain itu, ditampilkan AIC dan pseudo-R2 McFadden sebagai ukuran pendukung.

Hipotesis uji Hosmer-Lemeshow:

- H0: model sesuai dengan data
- H1: model tidak sesuai dengan data


In [11]:
hosmer_lemeshow <- function(y, p, g = 10) {
  n <- length(y)
  grup <- ceiling(g * rank(p, ties.method = "first") / n)
  grup[grup < 1] <- 1
  grup[grup > g] <- g

  detail <- aggregate(
    cbind(y = y, p = p),
    by = list(grup = grup),
    FUN = function(z) c(sum = sum(z), mean = mean(z), n = length(z))
  )

  obs <- as.numeric(detail$y[, "sum"])
  exp <- as.numeric(detail$p[, "sum"])
  total <- as.numeric(detail$y[, "n"])

  hl_stat <- sum((obs - exp)^2 / (exp * (1 - exp / total)))
  df_hl <- g - 2
  p_value <- 1 - pchisq(hl_stat, df_hl)

  detail_out <- data.frame(
    grup = detail$grup,
    n = total,
    observed_1 = obs,
    expected_1 = exp,
    observed_rate = obs / total,
    expected_rate = exp / total
  )

  list(statistic = hl_stat, df = df_hl, p_value = p_value, detail = detail_out)
}

hl <- hosmer_lemeshow(analysis_data$akses_internet, analysis_data$peluang_akses_internet, g = 10)
cat("Statistik Hosmer-Lemeshow:", round(hl$statistic, 4), "\n")
cat("df:", hl$df, "\n")
cat("p-value:", fmt_p(hl$p_value), "\n")
cat("Keputusan:", label_keputusan(hl$p_value, alpha, "Model kurang sesuai", "Tidak ada bukti model tidak sesuai"), "\n")
cat("\nDetail grup Hosmer-Lemeshow:\n")
print(hl$detail)

pseudo_r2_mcfadden <- 1 - as.numeric(logLik(model_final) / logLik(model_null))
cat("\nAIC model akhir:", AIC(model_final), "\n")
cat("Pseudo-R2 McFadden:", round(pseudo_r2_mcfadden, 4), "\n")


Statistik Hosmer-Lemeshow: 23.2748 


df: 8 


p-value: 0.0030 


Keputusan: Model kurang sesuai 



Detail grup Hosmer-Lemeshow:


   grup    n observed_1  expected_1 observed_rate expected_rate
1     1 3143          0    1.146452   0.000000000  0.0003647638
2     2 3143          5   11.941827   0.001590837  0.0037994994
3     3 3143         40   63.831326   0.012726694  0.0203090443
4     4 3143        238  225.655254   0.075723831  0.0717961355
5     5 3143        547  527.934682   0.174037544  0.1679715821
6     6 3143       1016  974.295581   0.323258034  0.3099890489
7     7 3143       1546 1571.818637   0.491886732  0.5001013798
8     8 3143       2230 2215.131632   0.709513204  0.7047825746
9     9 3143       2764 2774.731310   0.879414572  0.8828289247
10   10 3144       3005 3024.513299   0.955788804  0.9619953240



AIC model akhir: 20379.61 


Pseudo-R2 McFadden: 0.5056 


<!-- result-narrative -->

### Hasil Uji Kesesuaian Model untuk Laporan

Uji Hosmer-Lemeshow menghasilkan statistik sebesar **23,2748** dengan df **8** dan p-value **0,0030**. Karena p-value `<0,05`, H0 ditolak, sehingga terdapat indikasi bahwa model belum sepenuhnya sesuai dengan data berdasarkan pengelompokan peluang prediksi.

| Ukuran | Nilai |
|---|---:|
| Statistik Hosmer-Lemeshow | 23,2748 |
| df | 8 |
| p-value | 0,0030 |
| AIC model akhir | 20.379,61 |
| Pseudo-R2 McFadden | 0,5056 |

Interpretasi yang disarankan: model memiliki kemampuan penjelasan yang cukup kuat berdasarkan pseudo-R2 McFadden dan AIC yang jauh lebih rendah dibanding model kosong. Namun, hasil Hosmer-Lemeshow menunjukkan bahwa kalibrasi model belum sempurna, sehingga interpretasi model perlu disampaikan dengan hati-hati.


## 10. Analisis Ketepatan Klasifikasi

Klasifikasi dilakukan dengan ambang peluang `0.5`. Jika peluang prediksi `>= 0.5`, responden diklasifikasikan memiliki akses internet (`1`); jika kurang dari `0.5`, diklasifikasikan tidak memiliki akses internet (`0`).


In [12]:
threshold <- 0.5
predicted_class <- ifelse(analysis_data$peluang_akses_internet >= threshold, 1, 0)
actual_class <- analysis_data$akses_internet

conf_matrix <- table(
  Aktual = factor(actual_class, levels = c(0, 1)),
  Prediksi = factor(predicted_class, levels = c(0, 1))
)
print(conf_matrix)

tn <- conf_matrix["0", "0"]
fp <- conf_matrix["0", "1"]
fn <- conf_matrix["1", "0"]
tp <- conf_matrix["1", "1"]

akurasi <- (tp + tn) / sum(conf_matrix)
sensitivitas <- tp / (tp + fn)
spesifisitas <- tn / (tn + fp)
presisi <- tp / (tp + fp)
f1_score <- 2 * presisi * sensitivitas / (presisi + sensitivitas)
baseline_accuracy <- max(prop.table(table(actual_class)))

metrik_klasifikasi <- data.frame(
  metrik = c("Akurasi", "Sensitivitas", "Spesifisitas", "Presisi", "F1-score", "Baseline accuracy"),
  nilai = c(akurasi, sensitivitas, spesifisitas, presisi, f1_score, baseline_accuracy)
)
metrik_klasifikasi$nilai <- round(metrik_klasifikasi$nilai, 4)
print(metrik_klasifikasi)


      Prediksi
Aktual     0     1
     0 17875  2165
     1  2516  8875


             metrik  nilai
1           Akurasi 0.8511
2      Sensitivitas 0.7791
3      Spesifisitas 0.8920
4           Presisi 0.8039
5          F1-score 0.7913
6 Baseline accuracy 0.6376


<!-- result-narrative -->

### Hasil Ketepatan Klasifikasi untuk Laporan

Dengan ambang klasifikasi 0,5, model menghasilkan confusion matrix berikut:

| Aktual / Prediksi | Prediksi 0 | Prediksi 1 |
|---|---:|---:|
| Aktual 0 | 17.875 | 2.165 |
| Aktual 1 | 2.516 | 8.875 |

Metrik klasifikasi model:

| Metrik | Nilai |
|---|---:|
| Akurasi | 0,8511 |
| Sensitivitas | 0,7791 |
| Spesifisitas | 0,8920 |
| Presisi | 0,8039 |
| F1-score | 0,7913 |
| Baseline accuracy | 0,6376 |

Akurasi model sebesar **85,11%**, lebih tinggi dibanding baseline accuracy sebesar **63,76%**. Hal ini menunjukkan bahwa model memberikan peningkatan ketepatan klasifikasi dibandingkan aturan sederhana yang selalu memilih kelas mayoritas. Spesifisitas sebesar **89,20%** menunjukkan model cukup baik dalam mengenali responden yang tidak memiliki akses internet, sedangkan sensitivitas sebesar **77,91%** menunjukkan model juga cukup baik dalam mengenali responden yang memiliki akses internet.


## Ringkasan Interpretasi

Cell berikut membantu membuat ringkasan otomatis dari hasil utama: prediktor signifikan, kelayakan model, dan performa klasifikasi.


In [13]:
cat("Prediktor akhir yang masuk model:\n")
print(significant_predictors)

cat("\nUji serentak model penuh p-value:", fmt_p(p_serentak), "\n")
cat("Uji Hosmer-Lemeshow model akhir p-value:", fmt_p(hl$p_value), "\n")
cat("Akurasi klasifikasi:", round(akurasi, 4), "\n")
cat("Baseline accuracy:", round(baseline_accuracy, 4), "\n")

cat("\nOdds ratio model akhir, diurutkan berdasarkan p-value:\n")
print(or_model_akhir[order(or_model_akhir$p_value), c("term", "OR", "CI_95_low", "CI_95_high", "p_value_fmt")])


Prediktor akhir yang masuk model:


[1] "usia"                   "skor_ekonomi_subjektif" "jenis_kelamin"         
[4] "status_perkawinan"      "pendidikan"             "mampu_baca_koran"      
[7] "punya_telepon_seluler"  "aktivitas_utama"       



Uji serentak model penuh p-value: <0.001 


Uji Hosmer-Lemeshow model akhir p-value: 0.0030 


Akurasi klasifikasi: 0.8511 


Baseline accuracy: 0.6376 



Odds ratio model akhir, diurutkan berdasarkan p-value:


                                   term         OR  CI_95_low  CI_95_high
2                                  usia  0.8979191  0.8939331   0.9019229
12              punya_telepon_selulerYa  7.8921144  6.8095336   9.1468042
5  status_perkawinanMenikah/berpasangan  0.3092030  0.2742725   0.3485821
4                jenis_kelaminPerempuan  0.6064792  0.5569135   0.6604562
3                skor_ekonomi_subjektif  1.2477332  1.1973845   1.3001990
10           pendidikanPerguruan tinggi 68.7861354 29.6625598 159.5119385
6       status_perkawinanPernah menikah  0.5054274  0.4075315   0.6268395
14 aktivitas_utamaMengurus rumah tangga  0.7385350  0.6701846   0.8138563
11                   mampu_baca_koranYa  5.8386433  3.1571504  10.7976345
9               pendidikanSMA/sederajat 10.8192942  4.6869937  24.9748846
8               pendidikanSMP/sederajat  3.0002885  1.2981313   6.9343760
13            aktivitas_utamaBersekolah  0.8340425  0.6934943   1.0030751
15 aktivitas_utamaLainnya/tidak bekerj

<!-- result-narrative -->

## Draf Ringkasan Hasil untuk Artikel/Laporan

Berdasarkan data IFLS akses internet sebanyak 31.431 responden, proporsi responden yang memiliki akses internet adalah 36,24%, sedangkan 63,76% lainnya tidak memiliki akses internet. Analisis dilakukan menggunakan regresi logistik biner dengan `akses_internet` sebagai variabel respon. Variabel `kelompok_usia` dan `ekonomi_subjektif` tidak digunakan sesuai ketentuan, sedangkan variabel identitas tidak dimasukkan sebagai prediktor.

Hasil uji independensi menunjukkan bahwa seluruh prediktor yang dianalisis memiliki hubungan signifikan dengan akses internet. Hasil uji regresi logistik bivariat juga menunjukkan bahwa masing-masing prediktor signifikan terhadap akses internet dengan p-value `<0,001`. Pemeriksaan asosiasi antar-prediktor tidak menemukan pasangan variabel dengan nilai asosiasi `>=0,70`, sehingga tidak terdapat indikasi hubungan antar-prediktor yang terlalu tinggi berdasarkan batas tersebut.

Pada model regresi logistik multivariat, uji serentak menunjukkan bahwa model penuh signifikan dengan p-value `<0,001`. Uji individu berbasis joint Wald test juga menunjukkan bahwa seluruh prediktor signifikan dalam model penuh, sehingga model akhir memuat `usia`, `skor_ekonomi_subjektif`, `jenis_kelamin`, `status_perkawinan`, `pendidikan`, `mampu_baca_koran`, `punya_telepon_seluler`, dan `aktivitas_utama`.

Interpretasi odds ratio menunjukkan bahwa kepemilikan telepon seluler, kemampuan membaca koran, tingkat pendidikan yang lebih tinggi, dan skor ekonomi subjektif yang lebih tinggi cenderung meningkatkan odds memiliki akses internet. Sebaliknya, usia yang lebih tinggi, berjenis kelamin perempuan, status menikah/berpasangan atau pernah menikah, serta aktivitas utama mengurus rumah tangga cenderung menurunkan odds memiliki akses internet dibanding kategori referensinya.

Model akhir memiliki AIC sebesar 20.379,61 dan pseudo-R2 McFadden sebesar 0,5056. Namun, uji Hosmer-Lemeshow menghasilkan p-value 0,0030, sehingga terdapat indikasi bahwa model belum sepenuhnya sesuai dengan data dari sisi kalibrasi. Dari sisi klasifikasi, model menghasilkan akurasi sebesar 85,11%, sensitivitas 77,91%, spesifisitas 89,20%, presisi 80,39%, dan F1-score 79,13%. Akurasi ini lebih tinggi daripada baseline accuracy sebesar 63,76%, sehingga model memiliki performa klasifikasi yang lebih baik dibanding klasifikasi berdasarkan kelas mayoritas saja.
